# 1. Data Loading - Binance

In [ ]:
pip install python-binance

In [ ]:
import pandas as pd
import numpy as np
import requests
import time
import os
from binance.client import Client
client = Client()  # или Client(api_key, api_secret)

# All dates are fixed manually — we do NOT use datetime.now().

START_DATE = "2021-01-01"
END_DATE = "2026-07-29"

DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

print(f"Loading data from {START_DATE} to {END_DATE}")

In [ ]:
# Binance data loader function

In [ ]:
def get_binance_prices(symbol, start_date, end_date):
    """
    Download daily closing prices for one asset from Binance public API
    using the python-binance library.
    """
    klines = client.get_historical_klines(
        symbol,
        Client.KLINE_INTERVAL_1DAY,
        start_str=start_date,
        end_str=end_date,
    )

    df = pd.DataFrame(klines, columns=[
        "open_time", "open", "high", "low", "close", "volume",
        "close_time", "quote_asset_volume", "trades",
        "taker_buy_base", "taker_buy_quote", "ignore"
    ])

    series = pd.Series(
        data=df["close"].astype(float).values,
        index=pd.to_datetime(df["open_time"], unit="ms"),
        name=symbol,
    )
    return series

In [ ]:
#Binance data loader function + saving 

In [ ]:
assets = {
    "BTC-USD": "BTCUSDT",
    "ETH-USD": "ETHUSDT",
    "BNB-USD": "BNBUSDT",
    "SOL-USD": "SOLUSDT",
    "XRP-USD": "XRPUSDT",
    "DOGE-USD": "DOGEUSDT",
}

price_columns = {}

for asset_name, binance_symbol in assets.items():
    print(f"Downloading {asset_name}...")
    price_columns[asset_name] = get_binance_prices(binance_symbol, START_DATE, END_DATE)

# Combine all six price series into one table (rows = dates, columns = assets)
prices = pd.DataFrame(price_columns)

# Some assets may have slightly different daily timestamps (e.g. 00:00:00.001
# vs 00:00:00.000), so we normalize the index to just the date part.
prices.index = prices.index.normalize()

print(f"\nFinal table shape: {prices.shape}")
print(f"Date range: {prices.index.min().date()} to {prices.index.max().date()}")

prices.tail()

In [ ]:
prices.to_csv(f"{DATA_DIR}/prices.csv", index_label="date")
print("Saved to data/prices.csv")

# 2.Data Quality Checks 

In [ ]:
# Check 1: are there any missing values?
print("Missing values per asset:")
print(prices.isna().sum())

In [ ]:
# Check 2: are there any gaps in the daily calendar?
# Crypto trades 24/7, so unlike stocks, we expect ZERO missing calendar days.
all_expected_days = pd.date_range(prices.index.min(), prices.index.max(), freq="D")
missing_days = all_expected_days.difference(prices.index)

print(f"Missing calendar days: {len(missing_days)}")
if len(missing_days) > 0:
    print(missing_days)

In [ ]:
# Check 3: any zero or negative prices? (would indicate a data error)
invalid_prices = (prices <= 0).sum()
print("Zero or negative prices per asset:")
print(invalid_prices)

In [ ]:
# Check 4: any duplicate dates in the index?
print(f"Duplicate dates: {prices.index.duplicated().sum()}")

In [ ]:
# Check 5: visual inspection — do the price charts look reasonable?
prices.plot(figsize=(14, 8), subplots=True, layout=(3, 2), title="Daily Closing Prices")

# No missing values or duplicate dates were found. All 6 assets had continuous trading history over the selected period. Visual inspection did not reveal any anomalies such as stale prices or gaps in the data.

# Log_returns

In [ ]:
# Log returns are the standard input for risk metrics (VaR, volatility, etc.)
log_returns = np.log(prices / prices.shift(1)).dropna()

log_returns.to_csv(f"{DATA_DIR}/log_returns.csv", index_label="date")
print("Saved to data/log_returns.csv")

log_returns.describe()

# 3. Data Quality Check: resource comparison(Binance vs CoinGecko)

In [ ]:
from dotenv import load_dotenv
load_dotenv()

COINGECKO_API_KEY = os.getenv("COINGECKO_API_KEY")

if COINGECKO_API_KEY is None:
    raise ValueError("COINGECKO_API_KEY not found. Add it to a .env file.")

In [ ]:
def get_coingecko_prices(coin_id, days=365):
    url = f"https://api.coingecko.com/api/v3/coins/{coin_id}/market_chart"
    params = {"vs_currency": "usd", "days": days}
    headers = {"x-cg-demo-api-key": COINGECKO_API_KEY}

    response = requests.get(url, params=params, headers=headers, timeout=10)
    response.raise_for_status()

    raw_prices = response.json()["prices"]  # список [timestamp_ms, price]

    dates = [pd.to_datetime(p[0], unit="ms").normalize() for p in raw_prices]
    values = [p[1] for p in raw_prices]

    series = pd.Series(data=values, index=dates, name=coin_id)
    series = series.groupby(series.index).last()
    return series

In [ ]:
coingecko_ids = {
    "BTC-USD": "bitcoin",
    "ETH-USD": "ethereum",
    "BNB-USD": "binancecoin",
    "SOL-USD": "solana",
    "XRP-USD": "ripple",
    "DOGE-USD": "dogecoin",
}

coingecko_columns = {}

for asset_name, coin_id in coingecko_ids.items():
    print(f"Downloading {asset_name} from CoinGecko...")
    coingecko_columns[asset_name] = get_coingecko_prices(coin_id)
    time.sleep(1.5)  # stay under the free-tier rate limit

coingecko_prices = pd.DataFrame(coingecko_columns)

coingecko_prices.to_csv(f"{DATA_DIR}/coingecko_prices.csv", index_label="date")
print("Saved to data/coingecko_prices.csv")

coingecko_prices.tail()

In [ ]:
# We compare only BTC first, as a simple example
btc_binance = prices["BTC-USD"]
btc_coingecko = coingecko_prices["BTC-USD"]

comparison = pd.DataFrame({
    "binance": btc_binance,
    "coingecko": btc_coingecko,
}).dropna()

comparison["diff_pct"] = (comparison["binance"] - comparison["coingecko"]) / comparison["coingecko"] * 100

print("Comparing prices on the SAME calendar date:")
print(f"Max absolute difference: {comparison['diff_pct'].abs().max():.2f}%")

In [ ]:
# The difference above is often large (10-20%), which is suspicious for two
# major exchanges. The likely reason: Binance records the closing price at
# the END of a UTC day, while CoinGecko's timestamp corresponds to the
# START of a UTC day. So Binance's "close of day X" is really closer in
# time to CoinGecko's "price on day X+1", not "price on day X".

# We test this by shifting the CoinGecko series forward by one day before
# comparing, and check whether the difference gets smaller.
btc_coingecko_shifted = btc_coingecko.shift(-1)  # shift forward by 1 day

comparison_shifted = pd.DataFrame({
    "binance": btc_binance,
    "coingecko": btc_coingecko_shifted,
}).dropna()

comparison_shifted["diff_pct"] = (
    (comparison_shifted["binance"] - comparison_shifted["coingecko"])
    / comparison_shifted["coingecko"] * 100
)

print("Comparing prices AFTER shifting CoinGecko forward by 1 day:")
print(f"Max absolute difference: {comparison_shifted['diff_pct'].abs().max():.2f}%")

In [ ]:
# If the shifted comparison gives a much smaller difference, our timing
# hypothesis is confirmed, and we apply the same 1-day shift to all assets
# for the final comparison table.
summary_rows = []

for asset_name in assets:
    binance_series = prices[asset_name]
    coingecko_series_shifted = coingecko_prices[asset_name].shift(-1)

    merged = pd.DataFrame({
        "binance": binance_series,
        "coingecko": coingecko_series_shifted,
    }).dropna()

    diff_pct = (merged["binance"] - merged["coingecko"]) / merged["coingecko"] * 100

    summary_rows.append({
        "asset": asset_name,
        "days_compared": len(merged),
        "mean_diff_pct": diff_pct.mean(),
        "max_abs_diff_pct": diff_pct.abs().max(),
    })

comparison_summary = pd.DataFrame(summary_rows).set_index("asset")
comparison_summary.to_csv(f"{DATA_DIR}/binance_vs_coingecko_summary.csv")

print(comparison_summary.round(2))

# Conclusion

# Prices were sourced from Binance and cross-checked against CoinGecko(free tier limits history to 365 days, so verification covers only  the most recent year).

# A naive comparison on the same calendar date showed differences up to 13.89%, which was too large to be normal cross-exchange noise.  Investigation revealed that the two sources record the "daily price" at different points in time.

# After aligning timestamps with a 1-day shift, the mean difference  across all 6 assets dropped to within ±0.2%, with maximum single-day differences up to 5.21%. The residual maximum differences might reflect normal cross-exchange dispersion (CoinGecko aggregates prices across multiple exchanges, while this analysis uses Binance as a single reference exchange).

# Decision: documenting this as a known data limitation rather than investigating further, since:
# 1. Binance is used as the single primary source for all metrics in this project — CoinGecko is only a sanity check.
# 2. The affected window is a smaller window (just a year of CoinGecko data) while the overall is ~5.5 year analysis period.
# 3. Root-causing an external API's internal caching behavior is outside the scope of this project.